# Task 6 · SFT only
Clone the shared semeval repository and install the package in `task 6/`.
No code ZIP or local dataset upload. Edit REPO_URL / GIT_REF in the cell below.
Use Runtime → Change runtime type → GPU for training. The transfer notebook can run on CPU.


In [ ]:
# Repository settings: editable in the notebook.
REPO_URL = 'https://github.com/iamtro-2006/semeval.git'
GIT_REF = 'main'  # branch, tag, or commit
CHECKOUT = '/content/semeval'
PROJECT_SUBDIR = 'task 6'

from pathlib import Path
import os, subprocess, sys
import codecs

def stream_run(cmd, check=True, cwd=None):
    # Route child output through IPython stdout, rather than its hidden OS file descriptor.
    env = dict(os.environ, PYTHONUNBUFFERED='1')
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env,
                               cwd=cwd if cwd is not None else globals().get('PROJECT_ROOT'))
    decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
    try:
        while True:
            chunk = os.read(process.stdout.fileno(), 4096)
            if not chunk:
                break
            sys.stdout.write(decoder.decode(chunk))
            sys.stdout.flush()
        sys.stdout.write(decoder.decode(b'', final=True))
        sys.stdout.flush()
        result = subprocess.CompletedProcess(cmd, process.wait())
    except BaseException:
        process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    finally:
        process.stdout.close()
    if check:
        result.check_returncode()
    return result

checkout = Path(CHECKOUT).expanduser()
if not checkout.is_absolute():
    raise ValueError('CHECKOUT must be absolute, e.g. /content/semeval')
checkout = checkout.resolve()
if not checkout.exists():
    checkout.parent.mkdir(parents=True, exist_ok=True)
    stream_run(['git', 'clone', REPO_URL, str(checkout)], check=True, cwd=checkout.parent)
else:
    actual = subprocess.check_output(['git', '-C', str(checkout), 'remote', 'get-url', 'origin'], text=True).strip()
    if actual != REPO_URL:
        raise ValueError('CHECKOUT belongs to a different repository; choose another CHECKOUT')
    dirty = subprocess.check_output(['git', '-C', str(checkout), 'status', '--porcelain'], text=True)
    if dirty.strip():
        raise RuntimeError('Checkout has edits; preserve your changes or choose a fresh CHECKOUT before switching revision')
stream_run(['git', '-C', str(checkout), 'fetch', 'origin', GIT_REF], check=True, cwd=checkout.parent)
stream_run(['git', '-C', str(checkout), 'checkout', '--detach', 'FETCH_HEAD'], check=True, cwd=checkout.parent)
PROJECT_ROOT = (checkout / PROJECT_SUBDIR).resolve()
SRC = PROJECT_ROOT / 'src'
if not (SRC / 'mmcultureqa/__init__.py').is_file():
    raise FileNotFoundError(f'Project package not found: {SRC / "mmcultureqa"}')
os.chdir(PROJECT_ROOT)
# Editable installation alone does not refresh sys.path in an already running kernel.
# Export PYTHONPATH too, so child Python processes and workers use this same checkout.
sys.path[:] = [str(SRC)] + [p for p in sys.path if p != str(SRC)]
python_paths = [p for p in os.environ.get('PYTHONPATH', '').split(os.pathsep) if p and p != str(SRC)]
os.environ['PYTHONPATH'] = os.pathsep.join([str(SRC), *python_paths])
import importlib
importlib.invalidate_caches()
for name in list(sys.modules):
    if name == 'mmcultureqa' or name.startswith('mmcultureqa.'):
        del sys.modules[name]
stream_run([sys.executable, '-m', 'pip', 'install', '--no-deps', '--no-build-isolation', '-e', str(PROJECT_ROOT)], check=True)
from mmcultureqa.common import ROOT as PACKAGE_ROOT
if PACKAGE_ROOT.resolve() != PROJECT_ROOT:
    raise ImportError(f'mmcultureqa loaded from {PACKAGE_ROOT}, expected {PROJECT_ROOT}')
stream_run([sys.executable, '-c', 'import mmcultureqa; print("Child package:", mmcultureqa.__file__)'])
print('Project:', Path.cwd())
print('Commit:', subprocess.check_output(['git', '-C', str(checkout), 'rev-parse', 'HEAD'], text=True).strip())


## Install GPU environment
Restart the runtime if requested, then rerun clone/setup. GPU training has not been run on the CPU-only development machine.


In [ ]:
INSTALL_VLLM = True  # optional backend for BF16 final/test inference; False for GRPO QLoRA
requirements = 'requirements-vllm.txt' if INSTALL_VLLM else 'requirements.txt'
stream_run([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT_ROOT / requirements)])
# Colab may have unrelated preinstalled-package conflicts; print the full report.
stream_run([sys.executable, '-m', 'pip', 'check'], check=False)
stream_run(['nvidia-smi'])


In [ ]:
# Fail early on dependency/import errors before downloading data or starting training.
import subprocess, sys
stream_run([sys.executable, '-c', """
import sys
from importlib.metadata import version
for name in ('numpy', 'torch', 'transformers', 'peft', 'trl', 'ms-swift', 'bitsandbytes'):
    print(name, version(name), flush=True)
from transformers.modeling_utils import PreTrainedModel
from peft import PeftModel
from swift.llm import TrainArguments, RLHFArguments
import qwen_vl_utils
import decord
from bitsandbytes.nn import Linear4bit
from swift.llm import PtEngine
import mmcultureqa.common
import mmcultureqa.data.prepare
import mmcultureqa.training.sft
import mmcultureqa.training.grpo
import mmcultureqa.training.curves
import mmcultureqa.inference.predict
import mmcultureqa.evaluate
from swift.llm.train.sft import SwiftSft
from swift.llm.train.rlhf import SwiftRLHF
from swift.llm import InferRequest, RequestConfig
if sys.argv[1] == '1':
    print('vllm', version('vllm'), flush=True)
    from vllm import LLM
    from swift.llm import VllmEngine
from swift.plugin import metric_mapping, orms, ORM
from swift.utils import Serializer
print('Training imports OK', flush=True)
""", '1' if INSTALL_VLLM else '0'])


## MMCQA folder and authentication
Only ROOT_FOLDER_ID selects storage. Paste the ID or URL of your MMCQA folder.
RCLONE_CONFIG_PATH points to your authenticated rclone.conf; this is not a storage root.
All data, adapters, checkpoints and results are addressed relative to the selected ID.
If your config is on mounted Drive, run the optional mount cell first.


In [ ]:
# Optional, only to access a config file stored on Drive:
# from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
ROOT_FOLDER_ID = '1kcdByZiSJzGSoEtY9q0OEhsZwiTZkxpV'  # ID or URL of the MMCQA folder
RCLONE_CONFIG_PATH = '/content/rclone.conf'
REMOTE = 'iamtro'  # Google Drive remote in rclone.conf


In [ ]:
stream_run(['sudo', 'apt-get', '-qq', 'update'])
stream_run(['sudo', 'apt-get', '-qq', 'install', '-y', 'rclone'])
import configparser, re, tempfile
from urllib.parse import urlparse
config_source = Path(RCLONE_CONFIG_PATH).expanduser()
if not config_source.is_absolute():
    raise ValueError('RCLONE_CONFIG_PATH must be absolute, e.g. /content/rclone.conf')
if not config_source.is_file():
    raise FileNotFoundError(f'Upload your authenticated rclone.conf or mount Drive: {config_source}')
sections = configparser.RawConfigParser()
sections.read(config_source, encoding='utf-8-sig')
remotes = [name for name in sections.sections() if sections[name].get('type') == 'drive']
if not REMOTE:
    if len(remotes) != 1:
        raise ValueError(f'Select REMOTE from Google Drive sections: {remotes}')
    REMOTE = remotes[0]
if REMOTE not in remotes:
    raise ValueError('REMOTE must be a Google Drive section')
root_id = ROOT_FOLDER_ID.strip()
if root_id.startswith(('https://', 'http://')):
    match = re.search(r'/folders/([^/?]+)', urlparse(root_id).path)
    root_id = match.group(1) if match else ''
if not re.fullmatch(r'[A-Za-z0-9_-]+', root_id):
    raise ValueError('Enter the ID or folder URL of MMCQA in ROOT_FOLDER_ID')
temporary_config = tempfile.TemporaryDirectory(prefix='mmcqa-rclone-')
config_path = Path(temporary_config.name) / 'rclone.conf'
config_path.write_bytes(config_source.read_bytes())
config_path.chmod(0o600)
rclone_base = ['rclone', '--config', str(config_path), '--drive-root-folder-id', root_id]
def cloud(relative):
    if relative.startswith('/') or '..' in Path(relative).parts:
        raise ValueError('Storage paths must be relative to MMCQA')
    return f'{REMOTE}:{relative}'
def copy(source, destination, *extra):
    stream_run(rclone_base + ['copy', str(source), str(destination),
                   '--transfers', '4', '--checkers', '8', '--progress', *extra], check=True)
stream_run(rclone_base + ['lsd', cloud(''), '--max-depth', '1'], check=True)
print('All storage is relative to MMCQA folder ID:', root_id)


## Prepare data on Colab SSD
The download location is automatically derived from ROOT_FOLDER_ID.


In [ ]:
USE_DRIVE_DATA = True
if USE_DRIVE_DATA:
    copy(cloud('task6-data/raw'), PROJECT_ROOT / 'data/raw')
else:
    stream_run([sys.executable, '-m', 'mmcultureqa.data.prepare', 'download'], check=True)
stream_run([sys.executable, '-m', 'mmcultureqa.data.prepare', 'prepare'], check=True)


## Experiment settings
Only ROOT_FOLDER_ID determines cloud storage. RUN_NAME groups your experiments under it.
Intermediate checkpoints stay on local SSD; only the selected final adapter uploads after successful training.
Uploads use explicit allowlists: adapter/processor, run settings, learning reports and validation results.
Optimizer states, exported training data, caches and temporary evaluation files stay local.
Local SSD is used for training and inference. If the runtime is lost before the final upload, local checkpoints cannot be recovered from Drive.
SFT defaults to BF16 LoRA. GRPO defaults to NF4 QLoRA with BF16 compute and native Transformers generation.
GRPO reuses the training Qwen model for generation, with no second vLLM Qwen and no CPU offload.
Both stages retain CUDA FP16 mmBERT scoring.
SFT's optional in-training generation monitor uses the native ms-swift Transformers path.
Final validation uses Transformers when USE_4BIT=True, otherwise vLLM.
These are unbenchmarked GPU starting configurations; measure throughput and peak memory on Colab.


In [ ]:
import json
USE_4BIT = False
PREDICT_WITH_GENERATE = False  # default SFT: val loss + token accuracy; True: generation + mmBERT
RUN_FULL = True  # full training from YAML; False explicitly selects a one-step smoke run
RUN_NAME = 'experiment01'
if not re.fullmatch(r'[A-Za-z0-9_-]+', RUN_NAME):
    raise ValueError('RUN_NAME must contain only letters, numbers, underscores or hyphens')
RUNS = PROJECT_ROOT / 'outputs/runs' / RUN_NAME
RUNS.mkdir(parents=True, exist_ok=True)
SFT_STAGE = 'sft' if RUN_FULL else 'sft-smoke'
GRPO_STAGE = 'grpo' if RUN_FULL else 'grpo-smoke'
SFT = str(RUNS / SFT_STAGE)
GRPO = str(RUNS / GRPO_STAGE)
common = {'model.dtype': 'bfloat16', 'model.load_in_4bit': USE_4BIT,
          'training.predict_with_generate': PREDICT_WITH_GENERATE}

def upload_run(local_output, cloud_subdir):
    from mmcultureqa.storage import upload_filters
    copy(local_output, cloud(cloud_subdir), *upload_filters('run'))

def upload_predictions(local_output, cloud_subdir):
    from mmcultureqa.storage import upload_filters
    copy(local_output, cloud(cloud_subdir), *upload_filters('predictions'))

def run_command(module, config, overrides):
    config = Path(config)
    if not config.is_absolute():
        config = PROJECT_ROOT / config
    if not config.is_file():
        raise FileNotFoundError(f'Configuration not found: {config}')
    cmd = [sys.executable, '-m', module, '--config', str(config)]
    for key, value in overrides.items():
        cmd += ['--set', f'{key}={json.dumps(value)}']
    stream_run(cmd, check=True)

def require_adapter(folder):
    folder = Path(folder)
    if not (folder / 'adapter_config.json').is_file():
        raise FileNotFoundError(f'Adapter config missing: {folder}')
    if not any((folder / name).is_file() for name in ('adapter_model.safetensors', 'adapter_model.bin')):
        raise FileNotFoundError(f'Adapter weights missing: {folder}')

def run(module, config, overrides, local_output, cloud_subdir):
    run_command(module, config, overrides)
    require_adapter(Path(local_output) / 'final')
    upload_run(local_output, cloud_subdir)


## SFT only · ms-swift + PEFT
Default PREDICT_WITH_GENERATE=False: monitor teacher-forced loss and token accuracy.
Full training runs 3 epochs and exports the best validation checkpoint as final/.
True: native ms-swift generation evaluation with mmBERT P/R/F1 plugin; select best by F1.
Generation evaluation replaces the teacher-forced validation pass; it does not report val loss simultaneously.
Edit SFT_OVERRIDES for prompt and training parameters. For same-stage resume, set
RESUME_CHECKPOINT to checkpoint-N only if you manually backed it up to Drive; automatic uploads contain final adapters only.


In [ ]:
SFT_OVERRIDES = {}  # e.g. {'training.learning_rate': 0.0001, 'prompt_file': 'prompts/qa.txt'}
RESUME_CHECKPOINT = ''  # manually backed-up checkpoint-N only; final/ is not a resumable checkpoint
sft_args = {**common, 'training.output_dir': SFT}
if not RUN_FULL:
    sft_args.update({'data.limit': 8, 'training.max_steps': 1, 'training.eval_strategy': 'no',
                     'training.load_best_model_at_end': False, 'training.save_steps': 1})
if RESUME_CHECKPOINT:
    copy(cloud(f'task6-runs/{RUN_NAME}/{SFT_STAGE}'), SFT)
    sft_args['resume_from_checkpoint'] = str(Path(SFT) / RESUME_CHECKPOINT)
sft_args.update(SFT_OVERRIDES)
run('mmcultureqa.training.sft', 'configs/sft.yaml', sft_args, SFT, f'task6-runs/{RUN_NAME}/{SFT_STAGE}')


In [ ]:
if not (Path(SFT) / 'final/adapter_config.json').is_file():
    raise FileNotFoundError('No final SFT adapter was saved')
print('SFT adapter on Drive:', cloud(f'task6-runs/{RUN_NAME}/{SFT_STAGE}/final'))
print('Notebook 03: select the same ROOT_FOLDER_ID, SFT_RUN_NAME and INPUT_SFT_STAGE.')


## Learning curves before final evaluation
Plot native training/validation metrics against optimizer step, then upload the PNG to Drive.
Validation points appear only at completed evaluation intervals; no interpolation or smoothing.


In [ ]:
import sys
from pathlib import Path

from mmcultureqa.training.curves import plot_learning_curves

local_run = Path(SFT)
if not local_run.is_absolute():
    local_run = PROJECT_ROOT / local_run
plot_learning_curves(local_run, 'sft')
upload_run(local_run, f'task6-runs/{RUN_NAME}/{SFT_STAGE}')


## Final held-out validation score
Generate answers with the saved final adapter and score with the unchanged official scorer.
Default: ALL 1,000 validation examples, even after a one-step training smoke run.
Set VAL_LIMIT=4 only to smoke-test evaluation; subset results are explicitly marked.
BERTScore-F1 is the ranking metric, not exact-match accuracy. No weights are updated.


In [ ]:
VAL_LIMIT = None  # full validation; optionally 4 for an evaluation smoke test
validation_run = Path(SFT)
cmd = [sys.executable, '-m', 'mmcultureqa.evaluate', '--run-dir', str(validation_run)]
cmd += ['--backend', 'transformers' if USE_4BIT else 'vllm']
if VAL_LIMIT is not None:
    cmd += ['--limit', str(VAL_LIMIT)]
result = stream_run(cmd, check=False)
upload_run(validation_run, f'task6-runs/{RUN_NAME}/{SFT_STAGE}')
result.check_returncode()
